![A soccer pitch for an international match.](soccer-pitch.jpg)

# Tests d'hypothèses avec des matchs de football masculin et féminin

Vous travaillez comme journaliste sportif dans un grand média en ligne, spécialisé dans l'analyse et le reportage sur le football. Vous suivez depuis plusieurs années les matchs internationaux féminins et masculins, et votre intuition vous dit qu'il se marque plus de buts lors des matchs internationaux féminins que masculins. Voilà un excellent sujet d'enquête qui plaira à vos abonnés, mais vous devez mener un test statistique d'hypothèse rigoureux pour en avoir le cœur net.

En cadrant le projet, vous tenez compte du fait que le football a beaucoup évolué au fil des ans et que les performances varient fortement selon les tournois. Vous décidez donc de limiter l'analyse aux matchs officiels de la `FIFA World Cup` (hors qualifications) depuis le `2002-01-01`.

Vous constituez deux jeux de données contenant les résultats de tous les matchs internationaux officiels féminins et masculins depuis le XIXe siècle, collectés depuis une source en ligne fiable. Ces données sont stockées dans deux fichiers CSV : `women_results.csv` et `men_results.csv`.

La question à laquelle vous cherchez à répondre est :

> Marque-t-on plus de buts lors des matchs internationaux féminins que masculins ?

Vous retenez un **seuil de significativité de 10 %** et posez les hypothèses nulle et alternative suivantes :

$H_0$ : Le nombre moyen de buts marqués lors des matchs internationaux féminins est identique à celui des matchs masculins.

$H_A$ : Le nombre moyen de buts marqués lors des matchs internationaux féminins est supérieur à celui des matchs masculins.


In [1]:
# Start your code here!
import pandas as pd
import pingouin

# Charger les données
women = pd.read_csv("women_results.csv")
men = pd.read_csv("men_results.csv")

# Convertir la colonne date en datetime
women["date"] = pd.to_datetime(women["date"])
men["date"] = pd.to_datetime(men["date"])

# Filtrer : FIFA World Cup uniquement, depuis le 2002-01-01
women_subset = women[
    (women["date"] > "2002-01-01") &
    (women["tournament"] == "FIFA World Cup")
].copy()
men_subset = men[
    (men["date"] > "2002-01-01") &
    (men["tournament"] == "FIFA World Cup")
].copy()

# Créer la colonne total de buts marqués par match
women_subset["total_goals"] = women_subset["home_score"] + women_subset["away_score"]
men_subset["total_goals"] = men_subset["home_score"] + men_subset["away_score"]

# Test de Mann-Whitney U (distribution non normale -> test non parametrique)
# alternative="greater" car H_A : femmes > hommes
results_pg = pingouin.mwu(
    x=women_subset["total_goals"],
    y=men_subset["total_goals"],
    alternative="greater"
)

# Le nom de la colonne p-value varie selon la version de pingouin
pval_col = "p_val" if "p_val" in results_pg.columns else "p-val"
p_val = results_pg[pval_col].values[0]

# Décision au seuil de 10 %
if p_val <= 0.10:
    result = "reject"
else:
    result = "fail to reject"

result_dict = {"p_val": p_val, "result": result}
print(result_dict)


{'p_val': np.float64(0.005106609825443641), 'result': 'reject'}


## Interprétation des résultats

Le test de Mann-Whitney U (choisi car le nombre de buts par match suit une distribution asymétrique et non normale, ce qui exclut un test t classique) renvoie une **p-value d'environ 0.0051 (0.51 %)**.

Comme cette p-value est largement inférieure au seuil de signification retenu de **10 % (0.10)**, nous **rejetons l'hypothèse nulle H₀** en faveur de l'hypothèse alternative H_A.

**Conclusion :** au regard des matchs officiels de la FIFA World Cup joués depuis le 2002-01-01, le nombre moyen de buts marqués lors des matchs internationaux **féminins** est **statistiquement significativement supérieur** à celui des matchs **masculins**. Ce n'est donc pas une simple impression de journaliste : les données confirment l'intuition de départ avec un résultat robuste.

À noter : la p-value obtenue est si faible qu'elle resterait significative même à un seuil beaucoup plus strict (1 %), ce qui renforce la solidité de cette conclusion.
